In [1]:
! pip install torch


In [2]:
import torch


In [8]:
# when we true the requires_grad that mean pytorch know that we are going to calalute the grad of that tensor
x = torch.tensor(3.0,requires_grad=True)


In [9]:
y = x**2

In [10]:
print(x)
print(y)

tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)


In [11]:
y.backward()

In [14]:
x.grad

tensor(6.)

In [15]:
x = torch.tensor(3.0,requires_grad=True)

In [22]:
y = x**2 

In [23]:
z = torch.sin(y)

In [25]:
z.backward()

In [26]:
x.grad

tensor(-5.4668)

In [27]:
# but we cant find the grad of y node
y.grad

/var/folders/83/vkcpj0h51_1gwqn9b___8hh00000gn/T/ipykernel_89219/3432883849.py:2: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:498.)
  y.grad


### x -> sq -> y -> sin -> Z  <br>
#### so here Z is root node and x is leafnode <br>
#### we can only calclaute the grad of leaf node


In [52]:
import torch

# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

In [53]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))



In [ ]:
# Forwar pass
z = x * w + b
y_pred = torch.sigmoid(z)
print(y_pred)

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)
print(loss)






tensor(0.9988)
tensor(6.7012)


### we manuly first def it 

In [ ]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))


dy_pred_dz = y_pred * (1 - y_pred)

dz_dw = x  # dz/dw = x
dz_db = 1


dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db



In [ ]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")
# now we callaute the grad after that we need to update our w and b 
# but by doing this manualy it take to much time and it is only the one perceptron 
# what if we have multi nn


Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


## now using the Autograd 

In [59]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

In [72]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [73]:
z = x *w + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [74]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [75]:
loss = binary_cross_entropy_loss(y_pred,y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [76]:
loss.backward()

In [77]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


## grad cleing problem
-> if we run it again all the y and z and agin calclaute the grad it will add with our prev grad

In [ ]:
u = torch.tensor(2.0,requires_grad=True)

tensor(2., requires_grad=True)

In [94]:
## form here run all the cell again 
y = u**2

In [95]:
y.backward()

In [96]:
u.grad

tensor(4.)

In [93]:
# clearing grad
u.grad.zero_()

tensor(0.)

### but after the training we dont need to track the grad of it 
### or calculating the grad so we remove it to use the forward pass only

# disable gradient tracking

In [97]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [98]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [99]:
y.backward()

In [100]:
x.grad

tensor(4.)

In [101]:
# option 1 - requires_grad_(False)
# option 2 - detach()
# option 3 - torch.no_grad()

In [102]:
x.requires_grad_(False)

tensor(2.)

In [103]:
x

tensor(2.)

In [104]:
y = x ** 2

In [105]:
y

tensor(4.)

In [106]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [108]:
# sencod 

x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [ ]:
# here what we do we copy th all the value in z 
z = x.detach()
z

tensor(2.)

In [109]:
y = z **2

In [110]:
y

tensor(4.)

In [111]:
y = x **2

In [112]:
y

tensor(4., grad_fn=<PowBackward0>)

In [115]:
## 3rd method is with torch.no_grad_

with torch.no_grad():
    y = x ** 2
y

tensor(4.)